# <font color = "teal" > INITIALISING THE SETUP <font>

In [1]:
# Initialising the setup

sc

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
2,application_1685511621620_0003,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

<SparkContext master=yarn appName=livy-session-2>

In [2]:
from pyspark.sql import SparkSession

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
spark = SparkSession.builder.appName("Basics").getOrCreate()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
# Importing pyspark functions

from pyspark.sql import functions as F

# Import for typecasting columns
from pyspark.sql.types import IntegerType,BooleanType,DateType,FloatType,StringType
from pyspark.sql.types import ArrayType

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

 ## <font color='red'>Task 01: Reading the data </font>

In [5]:
# Reding the dataframe from a csv format file

raw_recipes_df = spark.read.options(inferSchema = True, header = True).csv("s3://upgrad-new-sbuck/RAW_recipes_cleaned.csv")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
# Checking the dataframe

raw_recipes_df.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+------+-------+--------------+-------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-------------+
|                name|    id|minutes|contributor_id|          submitted|                tags|           nutrition|n_steps|               steps|         description|         ingredients|n_ingredients|
+--------------------+------+-------+--------------+-------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-------------+
|arriba   baked wi...|137739|     55|         47892|2005-09-16 00:00:00|['60-minutes-or-l...|[51.5, 0.0, 13.0,...|     11|['make a choice a...|autumn is my favo...|['winter squash',...|            7|
|a bit different  ...| 31490|     30|         26278|2002-06-17 00:00:00|['30-minutes-or-l...|[173.4, 18.0, 0.0...|      9|['preheat oven to...|this recipe calls...|['prepared pizza ...|            6|


In [7]:
# Checking the schema of dataframe

raw_recipes_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- name: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- minutes: integer (nullable = true)
 |-- contributor_id: integer (nullable = true)
 |-- submitted: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- nutrition: string (nullable = true)
 |-- n_steps: integer (nullable = true)
 |-- steps: string (nullable = true)
 |-- description: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- n_ingredients: integer (nullable = true)

In [8]:
# Checking number of columns in the dataframe

len(raw_recipes_df.columns)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

12

In [9]:
# Assert commands to check the dataframe that we have just read.

assert raw_recipes_df.count() == 231637, "There is a mistake in reading the data."
assert len(raw_recipes_df.columns) == 12, "There is a mistake in reading the data."
assert raw_recipes_df.schema["minutes"].dataType == IntegerType(), "The data types have not been read correctly."
assert raw_recipes_df.schema["tags"].dataType == StringType(), "The data types have not been read correctly."
assert raw_recipes_df.schema["n_ingredients"].dataType == IntegerType(), "The data types have not been read correctly."

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

 ## <font color='red'>Task 02: Extracting individual features from the nutrition column.

In [10]:
# Performing string operation to remove square brackets.

from pyspark.sql.functions import regexp_replace

raw_recipes_df = raw_recipes_df.withColumn("nutrition", regexp_replace(raw_recipes_df["nutrition"], r"\[|\]", "") )

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
# Listing out all the nutrition columns.

nutrition_column_names = ['calories',
                          'total_fat_PDV',
                          'sugar_PDV',
                          'sodium_PDV',
                          'protein_PDV',
                          'saturated_fat_PDV',
                          'carbohydrates_PDV']

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
# splitting the nutrition string into seven individial values. 

from pyspark.sql.functions import col
from pyspark.sql.functions import split


nutrition_cols_split = raw_recipes_df.withColumn("split_nutrition", split(raw_recipes_df["nutrition"], ","))

# Using select function to convert the array split_nutrition into 7 different columns

raw_recipes_df = nutrition_cols_split.select(["*"] + [nutrition_cols_split.split_nutrition[i].alias(column_name) for i, column_name in enumerate(nutrition_column_names)])  

# Dropping nutrition column after spliting the values and storing it into an array named split_nutrition.
raw_recipes_df = raw_recipes_df.drop("nutrition")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
# Checking the dataframe schema once again

raw_recipes_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- name: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- minutes: integer (nullable = true)
 |-- contributor_id: integer (nullable = true)
 |-- submitted: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- n_steps: integer (nullable = true)
 |-- steps: string (nullable = true)
 |-- description: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- n_ingredients: integer (nullable = true)
 |-- split_nutrition: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- calories: string (nullable = true)
 |-- total_fat_PDV: string (nullable = true)
 |-- sugar_PDV: string (nullable = true)
 |-- sodium_PDV: string (nullable = true)
 |-- protein_PDV: string (nullable = true)
 |-- saturated_fat_PDV: string (nullable = true)
 |-- carbohydrates_PDV: string (nullable = true)

In [14]:
# Typecasting all the nutrition columns from string to float type

for x in nutrition_column_names:
    raw_recipes_df = raw_recipes_df.withColumn(x, F.col(x).cast(FloatType()))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
# Checking Schema after conversion of nutrition columns into float type.

raw_recipes_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- name: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- minutes: integer (nullable = true)
 |-- contributor_id: integer (nullable = true)
 |-- submitted: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- n_steps: integer (nullable = true)
 |-- steps: string (nullable = true)
 |-- description: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- n_ingredients: integer (nullable = true)
 |-- split_nutrition: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- calories: float (nullable = true)
 |-- total_fat_PDV: float (nullable = true)
 |-- sugar_PDV: float (nullable = true)
 |-- sodium_PDV: float (nullable = true)
 |-- protein_PDV: float (nullable = true)
 |-- saturated_fat_PDV: float (nullable = true)
 |-- carbohydrates_PDV: float (nullable = true)

In [16]:
# Checking Number of columns in the dataframe after making the above changes.

len(raw_recipes_df.columns)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

19

In [17]:
# Checking if the above steps are performed correctly with the help of assert function.

assert raw_recipes_df.schema["carbohydrates_PDV"].dataType == FloatType(), "Recheck your typecasting"
assert raw_recipes_df.collect()[123432][14] == 62.0, "The columns have not been split correctly."
assert raw_recipes_df.collect()[10000][12] == 60.400001525878906, "The columns have not been split correctly."

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## <font color='red'>Task 03: Standardizing the nutrition values </font>

In [18]:
# Standardizing the values in the nutrition columns.

for nutrition_col in nutrition_column_names:
    if nutrition_col != "calories":
        nutrition_per_100_cal_col = (nutrition_col
                                 .replace('_PDV','')
                                 +'_per_100_cal')
        raw_recipes_df = raw_recipes_df.withColumn(nutrition_per_100_cal_col, raw_recipes_df[nutrition_col]*100/col("calories"))
        
        # Performing a fill na operation to fill all the nulls with 0s.
        raw_recipes_df = raw_recipes_df.withColumn(nutrition_per_100_cal_col, col(nutrition_per_100_cal_col)).fillna(0)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [19]:
# Using Assert commands to check if the above operations are working properly.

# total fat check for id 28881
assert raw_recipes_df.filter("id == 28881").select('total_fat_per_100_cal').first()[0] == 0, "total_fat_per_100_cal for recipe 28881 should be 0"

# total fat check for id 112140
assert round(raw_recipes_df.filter("id == 112140").select('total_fat_per_100_cal').first()[0]) == 8, "total_fat_per_100_cal for recipe 112140 should be 8"

# checking for nulls
for c in ['total_fat_per_100_cal','sugar_per_100_cal','sodium_per_100_cal','protein_per_100_cal',
                          'saturated_fat_per_100_cal','carbohydrates_per_100_cal']:
    assert raw_recipes_df.select(F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c)).collect()[0][0] == 0, "There are Nulls in the data"

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [20]:
# Checking if the above code is working properly by checking the values and the calculations.

raw_recipes_df.select("sodium_per_100_cal", "sodium_PDV", "calories").show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------+----------+--------+
|sodium_per_100_cal|sodium_PDV|calories|
+------------------+----------+--------+
|               0.0|       0.0|    51.5|
|  9.80392191371621|      17.0|   173.4|
| 17.79095706884644|      48.0|   269.8|
| 0.543330607671212|       2.0|   368.1|
| 6.517427145874804|      23.0|   352.9|
|1.8726592117035106|       3.0|   160.2|
| 6.304176314800952|      24.0|   380.7|
|24.785939612438035|     275.0|  1109.5|
| 2.599044794330321|     111.0|  4270.8|
| 4.008541491878184|     107.0|  2669.3|
|               0.0|       0.0|    79.2|
|1.3622122779691264|      10.0|   734.1|
|14.922145525686739|      69.0|   462.4|
| 2.849905113327678|       9.0|   315.8|
|               0.0|       0.0|     8.2|
|24.597919347841934|      26.0|   105.7|
|11.702127659574469|      11.0|    94.0|
|1.7189514621651205|       4.0|   232.7|
| 7.334345080337101|      29.0|   395.4|
|0.5740528229154431|       1.0|   174.2|
+------------------+----------+--------+
only showing top

In [21]:
# Checking Schema After making the above changes.

raw_recipes_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- name: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- minutes: integer (nullable = true)
 |-- contributor_id: integer (nullable = true)
 |-- submitted: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- n_steps: integer (nullable = true)
 |-- steps: string (nullable = true)
 |-- description: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- n_ingredients: integer (nullable = true)
 |-- split_nutrition: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- calories: float (nullable = false)
 |-- total_fat_PDV: float (nullable = false)
 |-- sugar_PDV: float (nullable = false)
 |-- sodium_PDV: float (nullable = false)
 |-- protein_PDV: float (nullable = false)
 |-- saturated_fat_PDV: float (nullable = false)
 |-- carbohydrates_PDV: float (nullable = false)
 |-- total_fat_per_100_cal: double (nullable = false)
 |-- sugar_per_100_cal: double (nullable = false)
 |-- sodium_per_100_cal: double (nullabl

In [22]:
# Checking number of columns in the dataframe after making the above changes.

len(raw_recipes_df.columns)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

25

## <font color='red'>Task 04: Convert the tags column from a string to an array of strings </font>

In [23]:
# Checking values in tags columns.

raw_recipes_df.select("tags").show(truncate = False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|tags                                                                                                                                                                                                                                                                                                                                                                                                                                  |
+-----------------------------------------------------------------------------------------------------------------------------------------------------

In [24]:
# Remove [ ] ' punctuation marks from the tags column.
raw_recipes_df = raw_recipes_df.withColumn("tags", regexp_replace(raw_recipes_df["tags"], r"[\[\]']", ""))

# Removing empty spaces from the tags column.
raw_recipes_df = raw_recipes_df.withColumn("tags", regexp_replace(raw_recipes_df["tags"], "\\s+", ""))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [25]:
# Checking values in tags column after making the above changes.

raw_recipes_df.select("tags").show(truncate = False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|tags                                                                                                                                                                                                                                                                                                                                       |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [26]:
# Splitting the values in the tags column.

raw_recipes_df = raw_recipes_df.withColumn("tags", split(raw_recipes_df["tags"], ","))


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [27]:
# Checking schema of the tags column after performing the split function.

raw_recipes_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- name: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- minutes: integer (nullable = true)
 |-- contributor_id: integer (nullable = true)
 |-- submitted: timestamp (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- n_steps: integer (nullable = true)
 |-- steps: string (nullable = true)
 |-- description: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- n_ingredients: integer (nullable = true)
 |-- split_nutrition: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- calories: float (nullable = false)
 |-- total_fat_PDV: float (nullable = false)
 |-- sugar_PDV: float (nullable = false)
 |-- sodium_PDV: float (nullable = false)
 |-- protein_PDV: float (nullable = false)
 |-- saturated_fat_PDV: float (nullable = false)
 |-- carbohydrates_PDV: float (nullable = false)
 |-- total_fat_per_100_cal: double (nullable = false)
 |-- sugar_per_100_cal: double (nullable =

In [28]:
# Checking the values in the tags column after performing the split

raw_recipes_df.select("tags").show(truncate = False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|tags                                                                                                                                                                                                                                                                                                                                                                      |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [29]:
# Casting the nullability of the tags column to explicitly to true as it was false previously by default.

raw_recipes_df = raw_recipes_df.withColumn("tags", raw_recipes_df["tags"].cast(ArrayType(StringType(), True)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [30]:
# Assert code for checking if the above code is working correctly for the tags column.

assert raw_recipes_df.schema["tags"].dataType == ArrayType(StringType(), True), "You have not split the string into an array."
assert raw_recipes_df.collect()[2][5] == ['time-to-make','course', 'preparation', 'main-dish', 'chili', 'crock-pot-slow-cooker', 'dietary', 'equipment', '4-hours-or-less'], "Recheck your string cleaning and splitting operations."

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## <font color='red'>Task 05: Reading the second data file </font>

In [31]:
# Reading the RAW_interactions dataset

raw_ratings_df = (spark.read.csv("s3://upgrad-new-sbuck/RAW_interactions_cleaned.csv", 
                                 header=True, 
                                 inferSchema= True)
                  .withColumn("review_date",  F.col("date"))
                  .drop(F.col("date"))
                  )

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [32]:
# Checking the Schema of the raw_ratings dataset.

raw_ratings_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- user_id: integer (nullable = true)
 |-- recipe_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- review: string (nullable = true)
 |-- review_date: timestamp (nullable = true)

In [33]:
# Assert commands to check if the dataframe is loaded correctly.

assert raw_ratings_df.count() == 1132367, "There is a mistake in reading the data."
assert len(raw_ratings_df.columns) == 5, "There is a mistake in reading the data."

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [34]:
# Checking the top 5 rows of the raw_ratings_df dataframe.

raw_ratings_df.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+---------+------+--------------------+-------------------+
|user_id|recipe_id|rating|              review|        review_date|
+-------+---------+------+--------------------+-------------------+
|  38094|    40893|     4|Great with a sala...|2003-02-17 00:00:00|
|1293707|    40893|     5|So simple  so del...|2011-12-21 00:00:00|
|   8937|    44394|     4|This worked very ...|2002-12-01 00:00:00|
| 126440|    85009|     5|I made the Mexica...|2010-02-27 00:00:00|
|  57222|    85009|     5|Made the cheddar ...|2011-10-01 00:00:00|
+-------+---------+------+--------------------+-------------------+
only showing top 5 rows

In [35]:
# Performing the inner join of both the datasets i.e. joining raw_ratings_df and raw_recipes_df.

interaction_level_df = raw_ratings_df.join(raw_recipes_df,
                                           raw_recipes_df.id == raw_ratings_df.recipe_id,
                                           "inner"
                                           )

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [36]:
# Checking the number of rows and columns in the dataframe after performing the inner join.

(interaction_level_df.count(), len(interaction_level_df.columns))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

(1132367, 30)

In [37]:
# Assert Code to check if the join is performed correctly.

assert (interaction_level_df.count() ,len(interaction_level_df.columns)) == (1132367, 30), "The type of join is incorrect"

list1 = raw_ratings_df.select('recipe_id').collect()
list2 = raw_recipes_df.select('id').collect()
exclusive_set = set(list1)-set(list2)

assert len(exclusive_set) == 0, "There is a mistake in reading one of the two data files."

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## <font color='red'>Task 06:  Creating time-based features</font>


In [38]:
interaction_level_df = (interaction_level_df
                        .withColumn('submitted', F.col("submitted").cast(DateType())
                                   )
                        .withColumn('review_date', F.col("review_date").cast(DateType())
                                   )
                                             
                       )

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [39]:
interaction_level_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- user_id: integer (nullable = true)
 |-- recipe_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- review: string (nullable = true)
 |-- review_date: date (nullable = true)
 |-- name: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- minutes: integer (nullable = true)
 |-- contributor_id: integer (nullable = true)
 |-- submitted: date (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- n_steps: integer (nullable = true)
 |-- steps: string (nullable = true)
 |-- description: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- n_ingredients: integer (nullable = true)
 |-- split_nutrition: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- calories: float (nullable = false)
 |-- total_fat_PDV: float (nullable = false)
 |-- sugar_PDV: float (nullable = false)
 |-- sodium_PDV: float (nullable = false)
 |-- protein_PDV: float (nullable = false)


In [40]:
from pyspark.sql.functions import datediff, months_between

interaction_level_df = (interaction_level_df
                        .withColumn('days_since_submission_on_review_date',
                                     datediff(col("review_date"),col("submitted"))              
                                   )
                        .withColumn('months_since_submission_on_review_date',
                                     months_between(col("review_date"),col("submitted"))          
                                   )
                        .withColumn('years_since_submission_on_review_date',
                                     months_between(col("review_date"),col("submitted"))/12         
                                   )
                         )

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [41]:
# Checking if the above code has worked properly.

interaction_level_df.select('days_since_submission_on_review_date','months_since_submission_on_review_date','years_since_submission_on_review_date').show(truncate = False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------------------------+--------------------------------------+-------------------------------------+
|days_since_submission_on_review_date|months_since_submission_on_review_date|years_since_submission_on_review_date|
+------------------------------------+--------------------------------------+-------------------------------------+
|326                                 |10.70967742                           |0.8924731183333333                   |
|2                                   |0.06451613                            |0.005376344166666667                 |
|601                                 |19.77419355                           |1.6478494625                         |
|2074                                |68.16129032                           |5.680107526666667                    |
|2459                                |80.80645161                           |6.7338709675                         |
|1371                                |45.06451613                       

In [42]:
# We can use this data for the reference.

interaction_level_df.select("submitted", "review_date").show(truncate = False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+-----------+
|submitted |review_date|
+----------+-----------+
|2004-01-30|2004-12-21 |
|2008-08-18|2008-08-20 |
|2008-08-18|2010-04-11 |
|2008-08-18|2014-04-23 |
|2001-08-10|2008-05-04 |
|2001-08-10|2005-05-12 |
|2001-08-10|2005-05-17 |
|2006-10-04|2008-12-26 |
|2011-05-17|2011-05-27 |
|2011-05-17|2011-11-21 |
|2000-03-13|2002-09-17 |
|2005-07-01|2006-08-14 |
|2005-07-01|2008-11-02 |
|2005-07-01|2009-03-29 |
|2005-07-01|2010-02-07 |
|2005-07-01|2011-01-25 |
|2005-07-01|2012-02-24 |
|2005-07-01|2017-05-30 |
|2005-07-01|2005-07-04 |
|2003-10-20|2003-11-16 |
+----------+-----------+
only showing top 20 rows

In [43]:
# Assert code to check if we have performed the above task correctly. 

assert interaction_level_df.schema["days_since_submission_on_review_date"].dataType == IntegerType()

assert (interaction_level_df.filter((interaction_level_df.user_id == 428885) & (interaction_level_df.recipe_id == 335241))
                            .select('days_since_submission_on_review_date').collect()[0][0]) == 77
assert (interaction_level_df.filter((interaction_level_df.user_id == 2025676) & (interaction_level_df.recipe_id == 94265))
                            .select('months_since_submission_on_review_date').collect()[0][0]) == 153.22580645
assert (interaction_level_df.filter((interaction_level_df.user_id == 338588) & (interaction_level_df.recipe_id == 21859))
                            .select('years_since_submission_on_review_date').collect()[0][0]) == 4.564516129166667

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## <font color = "teal"> Saving the data we have created so far in a parquet file. <font>

In [44]:
interaction_level_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- user_id: integer (nullable = true)
 |-- recipe_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- review: string (nullable = true)
 |-- review_date: date (nullable = true)
 |-- name: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- minutes: integer (nullable = true)
 |-- contributor_id: integer (nullable = true)
 |-- submitted: date (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- n_steps: integer (nullable = true)
 |-- steps: string (nullable = true)
 |-- description: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- n_ingredients: integer (nullable = true)
 |-- split_nutrition: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- calories: float (nullable = false)
 |-- total_fat_PDV: float (nullable = false)
 |-- sugar_PDV: float (nullable = false)
 |-- sodium_PDV: float (nullable = false)
 |-- protein_PDV: float (nullable = false)


In [45]:
# Assert command to check if we have right number of rows and columns in the final dataset.

assert (interaction_level_df.count() ,len(interaction_level_df.columns) ) == (1132367, 33)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [46]:
# Writing the "interaction_level_df" dataset to the S3 folder.


interaction_level_df.write.mode('overwrite').parquet('s3://upgrad-new-sbuck/Data/interaction_level_df_processed.parquet')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…